In [ ]:
#separar y ordenar extracción de características
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import numpy as np

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
#from tensorflow.keras.models import load_model

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
#from sklearn.metrics import f1_score
#from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
#from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import classification_report
from sklearn.feature_selection import SelectKBest, chi2

from xgboost import XGBClassifier


In [ ]:
data=pd.read_csv("dataset_phishing.csv",sep=",")

In [ ]:
data

In [ ]:
data_mejorada = data.copy()

In [ ]:
data_mejorada

In [ ]:
data_mejorada["domain_in_title"].unique()

In [ ]:
# Invertir los valores de la columna 'ip': 0 a 1 y 1 a 0
data_mejorada['google_index'] = data['google_index'].apply(lambda x: 1 if x == 0 else 0)

# Invertir los valores de la columna 'domain_with_copyright': 0 a 1 y 1 a 0
data_mejorada['domain_with_copyright'] = data['domain_with_copyright'].apply(lambda x: 1 if x == 0 else 0)

# Invertir los valores de la columna 'https_token': 0 a 1 y 1 a 0
data_mejorada['https_token'] = data['https_token'].apply(lambda x: 1 if x == 0 else 0)

# Invertir los valores de la columna 'whois_registered_domain': 0 a 1 y 1 a 0
data_mejorada['whois_registered_domain'] = data['whois_registered_domain'].apply(lambda x: 1 if x == 0 else 0)

# Invertir los valores de la columna 'domain_in_title': 0 a 1 y 1 a 0
data_mejorada['domain_in_title'] = data['domain_in_title'].apply(lambda x: 1 if x == 0 else 0)

# Invertir los valores de la columna 'dns_record': 0 a 1 y 1 a 0
data_mejorada['dns_record'] = data['dns_record'].apply(lambda x: 1 if x == 0 else 0)

In [ ]:
data_mejorada

In [ ]:
import pandas as pd
import re
from urllib.parse import urlparse

# Función para verificar si un dominio contiene una dirección IP
def having_ip_address(url):
    match = re.search(
        r'(([01]?\d\d?|2[0-4]\d|25[0-5])\.([01]?\d\d?|2[0-4]\d|25[0-5])\.([01]?\d\d?|2[0-4]\d|25[0-5]))|'  # IPv4
        r'((0x[0-9a-fA-F]{1,2})\.(0x[0-9a-fA-F]{1,2})\.(0x[0-9a-fA-F]{1,2})\.(0x[0-9a-fA-F]{1,2}))|'  # IPv4 hex
        r'(?:[a-fA-F0-9]{1,4}:){7}[a-fA-F0-9]{1,4}',  # IPv6
        url
    )
    return 1 if match else 0

# Analizar la columna "url" y extraer el dominio, luego verificar si contiene una IP
data_mejorada['ip'] = data_mejorada['url'].apply(lambda x: having_ip_address(urlparse(x).netloc))

In [ ]:
data_mejorada[data_mejorada['ip'] == 1].sample(n=90)

In [ ]:
data_mejorada['status'] = data_mejorada['status'].apply(lambda x: 1 if x == 'phishing' else 0)

In [ ]:
data

In [ ]:
data_mejorada

In [ ]:
####################################################################################################################################################################################################################################################################################################################################################
################################################################################################################################################################################################################################################
################################################################################################################################################################################################################################################
################################################################################################################################################################################################################################################

In [ ]:
data_mejorada["status"].value_counts()

In [ ]:
### Análisis de Completitud ###

# Verificar y contar valores nulos en cada columna
valores_nulos = data_mejorada.isnull().sum()
porcentaje_nulos = (valores_nulos / len(data_mejorada)) * 100

# Crear un DataFrame con el conteo y porcentaje de valores nulos
nulos_df = pd.DataFrame({
    'Valores Nulos': valores_nulos,
    'Porcentaje de Nulos (%)': porcentaje_nulos
})

# Mostrar el DataFrame de valores nulos
print("Completitud del Dataset:")
print(nulos_df)

# Graficar el porcentaje de valores nulos por columna con una paleta de colores
plt.figure(figsize=(12, 6))
sns.barplot(
    x=nulos_df.index, 
    y=nulos_df['Porcentaje de Nulos (%)'], 
    color="skyblue"  # Usar un color fijo
)
plt.xticks(rotation=90)
plt.title("Porcentaje de Valores Nulos por Columna")
plt.xlabel("Columnas")
plt.ylabel("Porcentaje de Nulos (%)")
plt.show()

In [ ]:
### Análisis de Correctitud ###

# Describir el dataset para obtener estadísticas básicas (media, mediana, min, max, etc.)
print("Estadísticas Descriptivas:")
print(data_mejorada.describe())

# Detectar valores inusuales (ejemplo: valores negativos donde no deberían estar)
valores_incorrectos = {}
for col in data_mejorada.columns:
    if data_mejorada[col].dtype in ['float64', 'int64']:  # Solo columnas numéricas
        # Identificar valores negativos (ejemplo de valor incorrecto)
        incorrect_values = data_mejorada[data_mejorada[col] < 0]
        if not incorrect_values.empty:
            valores_incorrectos[col] = incorrect_values[col].values
            print(f"\nValores incorrectos en '{col}':")
            print(incorrect_values[col])

# Crear gráficos de distribución para cada columna numérica
num_cols = data_mejorada.select_dtypes(include=['float64', 'int64']).columns
num_plots = len(num_cols)
cols = 5  # Número de columnas de subplots
rows = math.ceil(num_plots / cols)  # Número de filas necesarias

plt.figure(figsize=(cols * 4, rows * 3))
for i, col in enumerate(num_cols, 1):
    plt.subplot(rows, cols, i)
    sns.histplot(data_mejorada[col], kde=True)
    plt.title(f"Distribución de '{col}'")
    plt.xlabel("")
plt.tight_layout()
plt.suptitle("Distribución y Posibles Valores Incorrectos en Características Numéricas", y=1.02)
plt.show()

In [ ]:
# Verificar valores únicos en la columna 'X'
print("Valores únicos en 'X':", data['ip'].unique())

# Contar valores únicos y verificar si hay valores nulos
print("Conteo de valores en 'X':")
print(data['ip'].value_counts(dropna=False))


# Revisar si hay nulos en 'X'
print("Cantidad de valores nulos en 'X':", data['ip'].isnull().sum())

In [ ]:
data_mejorada

In [ ]:
data_filtrada = data_mejorada.loc[:, data_mejorada.nunique() > 1].copy()

In [ ]:
data_filtrada

In [ ]:
data_filtrada.shape

In [ ]:
### Detección de Outliers ###

# Función para detectar outliers usando el método de Rango Intercuartílico (IQR)
def detectar_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers

# Detección y conteo de outliers para cada columna numérica
outliers_info = {}
for col in data_mejorada.select_dtypes(include=['float64', 'int64']).columns:
    outliers = detectar_outliers(data_mejorada, col)
    outliers_info[col] = len(outliers)
    print(f"{col}: {len(outliers)} outliers")

# Convertir el diccionario en un DataFrame para mejor visualización
outliers_df = pd.DataFrame(list(outliers_info.items()), columns=['Columna', 'Cantidad de Outliers'])
print("\nResumen de Outliers por Columna:")
print(outliers_df)

# Visualización de Outliers usando Boxplots
num_cols = data_mejorada.select_dtypes(include=['float64', 'int64']).columns
num_plots = len(num_cols)
cols = 5  # Número de columnas de subplots
rows = math.ceil(num_plots / cols)  # Número de filas necesarias

plt.figure(figsize=(cols * 4, rows * 3))
for i, col in enumerate(num_cols, 1):
    plt.subplot(rows, cols, i)
    sns.boxplot(x=data_mejorada[col])
    plt.title(f"Outliers en '{col}'")
    plt.xlabel("")
plt.tight_layout()
plt.suptitle("Detección de Outliers en Características Numéricas", y=1.02)
plt.show()


In [ ]:
data_mejorada

In [ ]:
data_filtrada

In [ ]:
data_filtrada = data_filtrada.drop(columns=["url"])

In [ ]:
data_filtrada

In [ ]:
# 1. Gráfico de correlación de cada característica con la variable objetivo 'estado'
correlaciones = data_filtrada.corr()['status'].abs().sort_values(ascending=False)
print("Correlación de características con la variable objetivo 'estado':")
print(correlaciones)

In [ ]:
# Gráfico de barras horizontal para visualizar la correlación con 'estado'
plt.figure(figsize=(15, 10))
correlaciones[1:].plot(kind='barh')  # Excluir 'estado' de sí mismo
plt.title("Correlación de Características con la Variable Objetivo 'estado'")
plt.xlabel("Valor Absoluto de Correlación")
plt.ylabel("Características")
plt.gca().invert_yaxis()  # Invertir el eje Y para que las características con mayor correlación estén arriba
plt.show()

# 2. Matriz de correlación de todas las características numéricas
plt.figure(figsize=(12, 10))
sns.heatmap(data_filtrada.corr(), annot=True, fmt=".2f", cmap="coolwarm", cbar_kws={'shrink': .8})
plt.title("Matriz de Correlación de Características Numéricas")
plt.show()


In [ ]:
def detectar_columnas_negativas(df):
    """
    Detecta columnas con valores negativos en un DataFrame.

    Args:
        df (pd.DataFrame): DataFrame a analizar.

    Returns:
        list: Lista de columnas que contienen valores negativos.
    """
    columnas_negativas = df.columns[(df < 0).any()].tolist()
    return columnas_negativas

# Ejemplo de uso
columnas_con_negativos = detectar_columnas_negativas(data_filtrada)
print("Columnas con valores negativos:", columnas_con_negativos)

In [ ]:
# Separar características (X) y variable objetivo (y)
X = data_filtrada.drop(columns=["status"])
y = data_filtrada['status']

In [ ]:
# Contar el número de instancias en cada clase
print("Distribución de la variable objetivo:")
print(y.value_counts())

# Calcular el porcentaje de cada clase
print("\nPorcentaje de cada clase:")
print(y.value_counts(normalize=True) * 100)

In [ ]:
# Entrenar un modelo de XGBoost
model_XGB = XGBClassifier(random_state=42, eval_metric='logloss')  # Quitar use_label_encoder
model_XGB.fit(X, y)

# Obtener las importancias de las características
importancias_XGB = model_XGB.feature_importances_

# Crear un DataFrame para ordenar las importancias
importancias_df_XGB = pd.DataFrame({
    'Característica': X.columns,
    'Importancia': importancias_XGB
}).sort_values(by='Importancia', ascending=False)

xgb_k = 46  # Número de características más importantes a mostrar
print(f"Las {xgb_k} características más importantes según XGBoost:")
print(importancias_df_XGB.head(xgb_k))

# Seleccionar las mejores características según XGBoost
mejores_columnas_XGB = importancias_df_XGB.head(xgb_k)['Característica'].values


In [ ]:
# Entrenar un Random Forest para calcular importancia
model_RF = RandomForestClassifier(random_state=42)
model_RF.fit(X, y)

# Obtener las importancias de las características
importancias_RF = model_RF.feature_importances_

# Crear un DataFrame para mostrar la importancia de cada característica
importancias_df_RF = pd.DataFrame({
    'Característica': X.columns,
    'Importancia': importancias_RF
}).sort_values(by='Importancia', ascending=False)

rf_k = 46

# Mostrar las características más importantes
print(f"Las {rf_k} características más importantes:")
print(importancias_df_RF.head(rf_k))

# Seleccionar las mejores características según Random Forest
mejores_columnas_RF = importancias_df_RF.head(rf_k)['Característica'].values

In [ ]:
# Asegurarse de que todas las características sean no negativas para chi2
X_copy = X.clip(lower=0)  # Chi2 requiere que los valores sean no negativos

# Seleccionar las mejores características según chi2
chi2_k = 46  # Número de características a seleccionar
selector = SelectKBest(chi2, k=chi2_k)
selector.fit(X_copy, y)  # Ajustar el selector a los datos

# Crear un DataFrame con las puntuaciones chi2
importancias_chi2 = selector.scores_
importancias_df_chi2 = pd.DataFrame({
    'Característica': X_copy.columns,
    'Importancia': importancias_chi2
}).sort_values(by='Importancia', ascending=False)

# Mostrar las características más importantes según chi2
print(f"Las {chi2_k} características más importantes según chi2:")
print(importancias_df_chi2.head(chi2_k))

# Seleccionar las mejores características según chi2
mejores_columnas_chi2 = importancias_df_chi2.head(chi2_k)['Característica'].values

In [ ]:
# Seleccionar las mejores características
X_new = X[mejores_columnas_XGB].drop(columns=["page_rank", "phish_hints", "domain_in_brand", "length_words_raw", "web_traffic", "safe_anchor", "ratio_extHyperlinks", "suspecious_tld", "shortening_service", "longest_words_raw", "ratio_extRedirection", "links_in_tags", "login_form", "ratio_extMedia", "external_favicon", "shortest_word_host", "avg_word_path", "ratio_extErrors", "nb_percent", "http_in_path", "nb_underscore", "nb_space", "nb_slash", "char_repeat", "shortest_word_path", "empty_title", "length_url", "longest_word_path"])

# Dividir en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.1, random_state=42)

# Escalar las características
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
pd.crosstab(data_mejorada["status"],data_mejorada["empty_title"],normalize="index").plot.bar()

In [ ]:
X_new

In [ ]:
# Definir el modelo de red neuronal

model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(24, activation='relu'),          # Aumenta a 18 neuronas en la primera capa
    Dropout(0.5),                          # Reduce dropout a 0.5
    Dense(12, activation='relu'),          # Incrementa a 9 neuronas en la segunda capa
    Dropout(0.5),                          # Aplica 0.5 dropout aquí también
    Dense(1, activation='sigmoid')         # Capa de salida para clasificación binaria
])

In [ ]:
model.summary()

In [ ]:
# Compilar el modelo con un learning rate ajustado
model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

# Implementar Early Stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenar el modelo
history = model.fit(X_train, y_train, 
                    epochs=100, 
                    batch_size=32, 
                    validation_split=0.2, 
                    callbacks=[early_stopping],
                    verbose=1)

# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nPrecisión en el conjunto de prueba: {accuracy:.4f}")

In [ ]:
y_pred_probs = model.predict(X_test)

In [ ]:
umbral=0.5

y_pred=y_pred_probs[:]>=umbral

fig, axes = plt.subplots(1,2)
fig.set_size_inches(10, 4)

matriz = confusion_matrix(y_test, y_pred)    
ax=axes[0]
sns.heatmap(matriz, annot=True, ax = ax, cmap="Blues",fmt="d");
ax.set_title("Umbral="+str(umbral)+"\nRec="+str(round(recall_score(y_test, y_pred),2))+
        " Acc="+str(round(accuracy_score(y_test, y_pred),2))+
        " Pre="+str(round(precision_score(y_test, y_pred),2)))
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Real')

fpt = 0
tpr = 0
fpr, tpr, thresholds = roc_curve(y_test, y_pred_probs) 
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, label='ROC curve (area = %0.2f)' % roc_auc)
axes[1].plot([0, 1], [0, 1], '--', label='No Skill')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse

def check_google_index(url):

    parsed_url = urlparse(url)
    domain = parsed_url.netloc

    # Construye la consulta con el dork site:
    query = f"site:{domain}"
    google_url = f"https://www.google.com/search?q={query}"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(google_url, headers=headers)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')

            # Busca elementos de resultados en la página
            search_results = soup.find_all('div', class_='g')  # Los resultados suelen estar en divs con clase "g"

            if search_results:
                return 1  # Si hay resultados, está indexado
            else:
                return 0  # No se encontraron resultados
        else:
            print("Error al realizar la consulta. Código de estado:", response.status_code)
            return 0  # Si no se puede realizar la consulta, asumimos que no está indexado
    except Exception as e:
        print("Error:", str(e))
        return 0  # Si ocurre un error, asumimos que no está indexado

# Ejemplo de uso
url = "https://csirt.gob.cl"
result = check_google_index(url)

if result == 1:
    print(f"El sitio {url} está indexado en Google.")
else:
    print(f"El sitio {url} NO está indexado en Google.")


In [ ]:
from datetime import datetime
from urllib.parse import urlparse
import whois
from publicsuffix2 import get_sld

def get_domain_age(url):

    try:
        # Parsear la URL para obtener el dominio
        parsed_url = urlparse(url)
        domain = parsed_url.netloc

        # Eliminar prefijo 'www.' si existe
        if domain.startswith("www."):
            domain = domain[4:]

        # Caso especial: dominios que terminan en 'gob.cl'
        if domain.endswith("gob.cl"):
            domain = "gob.cl"

        domain = get_sld(domain)

        # Obtener la información WHOIS del dominio
        whois_info = whois.whois(domain)
        creation_date = whois_info.creation_date

        # Manejar diferentes formatos de creation_date
        if creation_date is None:
            print(f"No se encontró la fecha de creación para el dominio '{domain}'.")
            return -1

        if isinstance(creation_date, list):  # Si es una lista, tomar el primer elemento
            creation_date = creation_date[0]

        if isinstance(creation_date, str):  # Convertir cadena a datetime
            creation_date = datetime.strptime(creation_date.split(" ")[0], "%Y-%m-%d")

        # Calcular la edad si creation_date es válido
        if isinstance(creation_date, datetime):
            domain_age = (datetime.now() - creation_date).days
            return domain_age
        else:
            print(f"El formato de 'creation_date' no es válido para '{domain}'.")
            return -1

    except Exception as e:
        print(f"Error al calcular la edad del dominio '{url}': {e}")
        return -1


# Probar con diferentes URLs
urls = [
    "https://www.pcfactory.cl/producto/47713-samsung-smartphone-galaxy-s23-ultra-256gb-8gb-green-liberado",
    "https://www.google.com",
    "https://canvasunab.unab.cl/inicio-alumnos/"
]

for url in urls:
    domain_age = get_domain_age(url)
    if domain_age != -1:
        print(f"La edad del dominio '{url}' es de {domain_age} días.")
    else:
        print(f"No se pudo calcular la edad del dominio '{url}'.")

In [ ]:
import dns.resolver

def dns_record(url):

    try:
        # Extraer el dominio de la URL
        domain = urlparse(url).netloc

        # Eliminar prefijos como 'www.' si existen
        if domain.startswith("www."):
            domain = domain[4:]

        # Caso especial: dominios que terminan en 'gob.cl'
        if domain.endswith("gob.cl"):
            domain = "gob.cl"

        domain = get_sld(domain)

        # Resolver registros NS del dominio
        nameservers = dns.resolver.resolve(domain, 'NS')

        # Verificar si se encontraron registros NS
        if len(nameservers) > 0:
            return 1  # Dominio válido con registros NS
        else:
            return 0  # No se encontraron registros NS
    except Exception as e:
        # En caso de error, asumimos que el dominio no tiene registros válidos
        #print(f"Error en la consulta DNS para {url}: {e}")
        return 0

In [ ]:
import requests
from bs4 import BeautifulSoup
import tldextract
import wordninja

def domain_in_title(url):
    """
    Verifica si alguna parte del nombre base del dominio de una URL aparece en el título de la página web.

    Parámetros:
    - url: URL completa de la página web (str).

    Retorno:
    - 1 si alguna palabra del nombre base del dominio aparece en el título.
    - 0 si ninguna palabra del nombre base del dominio aparece en el título.
    """
    try:
        # Extraer el nombre base del dominio de la URL
        extracted = tldextract.extract(url)
        base_domain = extracted.domain  # Solo el nombre del dominio principal (sin el TLD)

        # Dividir el nombre base del dominio en palabras usando wordninja
        palabras_base_domain = wordninja.split(base_domain)

        # Configurar el User-Agent para simular un navegador
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }

        # Obtener el contenido de la página web
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            #print(f"No se pudo acceder a la página: {url} (Código HTTP: {response.status_code})")
            return 0

        # Analizar el HTML de la página
        soup = BeautifulSoup(response.content, 'html.parser')
        title = soup.title.string if soup.title else ""

        # Verificar si alguna palabra del dominio aparece en el título
        for palabra in palabras_base_domain:
            if palabra.lower() in title.lower():
                return 1  # Coincidencia encontrada

        return 0  # Ninguna coincidencia encontrada

    except Exception as e:
        #print(f"Error en domain_in_title para la URL '{url}': {e}")
        return 0  # Asumir 0 en caso de error

In [ ]:
import re
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import requests
import whois

def count_nb_exclamation(url):
    return url.count('?')

def count_nb_www(url):
    return url.lower().count('www')

def ratio_digits_host(url):

    try:
        # Extraer el host de la URL
        parsed_url = urlparse(url)
        host = parsed_url.netloc

        # Contar dígitos en el host
        num_digitos = sum(c.isdigit() for c in host)
        longitud_host = len(host)

        # Evitar divisiones por cero
        if longitud_host == 0:
            return 0.000000

        # Calcular la proporción y redondear a 6 decimales
        return round(num_digitos / longitud_host, 6)
    except Exception as e:
        print(f"Error procesando la URL: {url}, {e}")
        return 0.000000
    
def nb_hyperlinks(url):

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }
    
    try:
        # Hacer la solicitud GET con el User-Agent
        response = requests.get(url, headers=headers, timeout=10)
        
        # Verificar el código de respuesta HTTP
        if response.status_code != 200:
            #print(f"Error HTTP al acceder a la URL: {response.status_code}")
            return 0

        # Parsear el contenido HTML
        dom = BeautifulSoup(response.text, 'html.parser')

        # Contar etiquetas <a> con href
        href_count = len(dom.find_all('a', href=True))
        
        # Contar etiquetas con src (como <img>, <script>, etc.)
        src_count = len(dom.find_all(src=True))
        
        #print(f"Enlaces encontrados en '{url}': {href_count}")
        #print(f"Recursos encontrados en '{url}': {src_count}")
        
        return href_count + src_count
    except requests.exceptions.RequestException as e:
        #print(f"Error al hacer la solicitud a la URL {url}: {e}")
        return 0
    except Exception as e:
        #print(f"Error procesando el DOM: {e}")
        return 0

def count_longest_word_path(url):

    try:
        # Extraer el path de la URL
        parsed_url = urlparse(url)
        path = parsed_url.path  # Obtiene el path de la URL

        # Dividir el path en segmentos por '/'
        segmentos = path.split('/')

        # Dividir segmentos con '.' en subpalabras y calcular la longitud máxima
        palabras = [subpalabra for segment in segmentos for subpalabra in segment.split('.') if subpalabra]

        # Calcular la longitud de la palabra más larga
        if palabras:
            return max(len(palabra) for palabra in palabras)
        else:
            return 0  # Si no hay palabras en el path
    except Exception as e:
        print(f"Error procesando la URL: {url}, {e}")
        return 0
    
def count_nb_percentage(url):
    return url.count('%')

def count_nb_hyphens(url):
    return url.count('-')

def count__nb_underscore(url):
    return url.count('_')

def count_nb_slash(url):
    return url.count('/')

def count_nb_dots(url):
    return url.count('.')

def count_nb_eq(url):
    return url.count('=')

def count_shortest_word_path(url):

    try:
        # Extraer el path de la URL
        parsed_url = urlparse(url)
        path = parsed_url.path  # Obtiene el path de la URL

        # Dividir el path en palabras (excluyendo cadenas vacías y extensiones)
        palabras = [segment for segment in path.split('/') if segment and '.' not in segment]

        # Calcular la longitud de la palabra más corta
        if palabras:
            return min(len(palabra) for palabra in palabras)
        else:
            return 0  # Si no hay palabras en el path
    except Exception as e:
        print(f"Error procesando la URL: {url}, {e}")
        return 0

def count_length_hostname(url):
    parsed_url = urlparse(url)
    hostname = parsed_url.hostname
    return len(hostname)

def ratio_digits_url(url):

    # Contar la cantidad de dígitos en la URL
    num_digitos = len(re.findall(r'\d', url))
    
    # Calcular la longitud total de la URL
    longitud_url = len(url)
    
    # Evitar división por cero
    if longitud_url == 0:
        return 0.000000
    
    # Calcular el ratio
    return round(num_digitos / longitud_url, 6)

def domain_with_copyright(url):

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }

    try:
        # Obtener el dominio base de la URL
        domain_match = re.search(r'://(www\.)?([^/]+)', url)
        if not domain_match:
            #print(f"No se pudo extraer el dominio de la URL: {url}")
            return 0
        domain = domain_match.group(2)

        # Descargar el contenido de la página
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            #print(f"Error HTTP al acceder a la URL: {response.status_code}")
            return 0
        
        content = response.text

        # Buscar símbolos de copyright, trademark o registered
        match = re.search(r'(\N{COPYRIGHT SIGN}|\N{TRADE MARK SIGN}|\N{REGISTERED SIGN}|&copy;|&trade;|&reg;)', content)
        
        if match:
            # Extraer contexto con un rango más amplio
            context_range = 300  # Incrementar el rango a ±200 caracteres
            _copyright = content[max(0, match.span()[0]-context_range):match.span()[0]+context_range]
            
            # Verificar si el dominio aparece cerca del símbolo
            if domain.lower() in _copyright.lower():
                #print(f"El dominio '{domain}' está cerca de un símbolo de copyright.")
                return 1  # El dominio está junto al símbolo
            else:
                #print(f"El dominio '{domain}' no está cerca de un símbolo de copyright.")
                #print(f"Contexto extraído:\n{_copyright}")
                return 0  # El dominio no está junto al símbolo
        else:
            print("No se encontraron símbolos de copyright, trademark o registered en la página.")
            return 0  # No se encontraron símbolos
    except requests.exceptions.RequestException as e:
        #print(f"Error al acceder a la URL {url}: {e}")
        return 0  # Considerar como sospechoso si no se puede acceder
    except Exception as e:
        #print(f"Error general procesando la URL {url}: {e}")
        return 0

def https_token(url):

    try:
        # Validar si la URL no es vacía o nula
        if not url or not isinstance(url, str):
            raise ValueError("La URL está vacía o no es válida.")

        # Parsear la URL para obtener el esquema
        scheme = urlparse(url).scheme.lower()  # Convertir a minúsculas para evitar inconsistencias

        # Retornar 1 si el esquema es HTTPS, 0 en caso contrario
        return 1 if scheme == 'https' else 0
    except Exception as e:
        #print(f"Error al analizar la URL '{url}': {e}")
        return 0  # Considerar como no seguro si ocurre un error.

def whois_registered_domain(url):

    try:

        # Extraer el dominio de la URL
        domain = urlparse(url).netloc

        # Eliminar prefijos como 'www.' si existen
        if domain.startswith("www."):
            domain = domain[4:]
        
        # Consultar WHOIS del dominio
        hostname = whois.whois(domain).domain_name
        # Si domain_name es una lista, buscar coincidencias en cada elemento
        if isinstance(hostname, list):
            for host in hostname:
                if re.search(host.lower(), domain):
                    return 1  # Coincidencia encontrada
            return 0  # Ninguna coincidencia encontrada
        else:
            # Verificar coincidencia directa si es un string
            if re.search(hostname.lower(), domain):
                return 1  # Coincidencia encontrada
            else:
                return 0  # Sin coincidencias
    except Exception as e:
        #print(f"Error al consultar WHOIS para '{domain}': {e}")
        return  0 # Considerar como no registrado en caso de error

def tld_in_subdomain(url):

    try:
        # Parsear la URL para obtener el hostname
        parsed_url = urlparse(url)
        hostname = parsed_url.hostname  # Ejemplo: 'www.com.example.com'

        if not hostname:
            #print(f"URL inválida: {url}")
            return 0

        # Dividir el hostname en partes
        parts = hostname.split('.')
        
        # Determinar el TLD y el subdominio
        if len(parts) > 2:  # Hay al menos un subdominio
            tld = parts[-1]  # Última parte como TLD
            subdomain = ".".join(parts[:-2])  # Todo antes del dominio principal y TLD
        else:  # No hay subdominio
            tld = parts[-1] if len(parts) > 1 else ""
            subdomain = ""

        # Verificar si el TLD aparece en el subdominio o coincide directamente
        if tld and subdomain and (tld in subdomain or subdomain == tld):
            return 1
        return 0

    except Exception as e:
        #print(f"Error en tld_in_subdomain para la URL '{url}': {e}")
        return 0  # Asumir 0 en caso de error

def prefix_suffix(url):
    if re.findall(r"https?://[^\-]+-[^\-]+/", url):
        return 1
    else:
        return 0


In [ ]:
pd.crosstab(data_mejorada["status"],data_mejorada["domain_with_copyright"],normalize="index").plot.bar()

In [ ]:
def extract_features_url(url):
    
    # Inicializar características
    features = {
        'google_index': 0,
        'nb_qm': 0, #nueva
        'nb_www': 0,
        'ratio_digits_host': 0.000000,
        'nb_hyperlinks': 0,
        #'longest_word_path': 0, 
        'nb_hyphens': 0,
        'domain_in_title': 0, #nueva
        #'nb_slash': 0,
        'nb_dots': 0,
        'nb_eq': 0,
        #'shortest_word_path': 0,
        'length_hostname': 0,
        'domain_age': 0,
        'ratio_digits_url': 0,
        'domain_with_copyright': 0,
        #'char_repeat': 0,
        'https_token': 0,
        'whois_registered_domain': 0,
        'tld_in_subdomain': 0,
        'dns_record': 0, #nueva
        'prefix_suffix': 0 #nueva
    }

    features['google_index'] = check_google_index(url) # Terminado

    features['nb_qm']

    features['nb_www'] = count_nb_www(url) # Terminado

    features['ratio_digits_host'] = ratio_digits_host(url) # Terminado

    features['nb_hyperlinks'] = nb_hyperlinks(url) # Pendiente ---

    #features['longest_word_path'] = count_longest_word_path(url) # Pendiente ---

    features['nb_hyphens'] = count_nb_hyphens(url) # Terminado

    #features['nb_slash'] = count_nb_slash(url) # Terminado

    features['domain_in_title'] = domain_in_title(url)

    features['nb_dots'] = count_nb_dots(url) # Terminado

    features['nb_eq'] = count_nb_eq(url) # Terminado

    #features['shortest_word_path'] = count_shortest_word_path(url) # Pendiente --

    features['length_hostname'] = count_length_hostname(url) # Terminado

    features['domain_age'] = get_domain_age(url) # Ajustar

    features['ratio_digits_url'] = ratio_digits_url(url) # Terminado

    features['domain_with_copyright'] = domain_with_copyright(url) # Terminado

    #features['char_repeat'] # Pendiente --

    features['https_token'] = https_token(url) # Terminado

    features['whois_registered_domain'] = whois_registered_domain(url) # Terminado

    features['tld_in_subdomain'] = tld_in_subdomain(url)

    features['dns_record'] = dns_record(url)

    features['prefix_suffix'] = prefix_suffix(url)

    return features

#url = "http://174.139.46.123/ap/signin?openid.pape.max_auth_age=0&amp;openid.return_to=https%3A%2F%2Fwww.amazon.co.jp%2F%3Fref_%3Dnav_em_hd_re_signin&amp;openid.identity=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&amp;openid.assoc_handle=jpflex&amp;openid.mode=checkid_setup&amp;key=a@b.c&amp;openid.claimed_id=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0%2Fidentifier_select&amp;openid.ns=http%3A%2F%2Fspecs.openid.net%2Fauth%2F2.0&amp;&amp;ref_=nav_em_hd_clc_signin"

url = "http://www.crestonwood.com/router"
url = "https://www.lider.cl/supermercado/category/Precios_WOW/Supermercado?ads_name=2024-11-21_Sli1_precios_wow_ID:22011&ads_position=Slide_1&ads_id=GR&slot_id=WC54OWB3R9BY&slot_name=home-banner-1-desktop"
url = "https://escuelamartinpescador.cl/blog/2020/09/06/minuta-de-colaciones/"
url = "https://escuelamartinpescador.cl/reglamento/"
url = "https://support.paramountplus.com/s/contactsupport?language=es_MX"
url = "https://csirt.gob.cl/"
url = "https://escuelamartinpescador.cl/"

features = extract_features_url(url)
for key, value in features.items():
    print(f"{key}: {value}")

In [ ]:
X_new

In [ ]:
def predecir_probabilidad_phishing_url(model, scaler, url, nombres_columnas):

    # Extraer las características de la URL
    caracteristicas = extract_features_url(url)
    
    # Convertir las características a un DataFrame de pandas con los nombres de columna
    caracteristicas_df = pd.DataFrame([caracteristicas], columns=nombres_columnas)
    
    # Escalar las características
    caracteristicas_escaladas = scaler.transform(caracteristicas_df)

    # Realizar la predicción
    probabilidad = model.predict(caracteristicas_escaladas)[0][0]  # Probabilidad de clase positiva (phishing)

    return probabilidad

# Ejemplo de uso
nombres_columnas = [
    'google_index', 'nb_qm', 'nb_www', 'ratio_digits_host', 'nb_hyperlinks',
    'nb_hyphens', 'domain_in_title', 'nb_dots', 'nb_eq', 'length_hostname',
    'domain_age', 'ratio_digits_url', 'domain_with_copyright', 'https_token',
    'whois_registered_domain', 'tld_in_subdomain', 'dns_record', 'prefix_suffix'
]

url_ejemplo = "https://www.youtube.com/watch?v=rNUAr5qbeV0&list=RDhk4iI8WYVd4&index=7"

# Calcular la probabilidad de phishing para una URL
probabilidad_phishing = predecir_probabilidad_phishing_url(model, scaler, url_ejemplo, nombres_columnas)
print(f"Probabilidad de phishing: {probabilidad_phishing:.2%}")

In [ ]:
X_new

In [ ]:
import numpy as np
import pandas as pd

def predecir_probabilidad_phishing(model, scaler, caracteristicas, nombres_columnas):
    """
    Realiza una predicción ingresando manualmente las 15 características
    y devuelve la probabilidad de que sea phishing.

    Parámetros:
    - model: Modelo entrenado.
    - scaler: Escalador utilizado para normalizar los datos.
    - caracteristicas: Lista o array con las X características en el orden correcto.
    - nombres_columnas: Lista de nombres de columnas utilizadas para entrenar el scaler.

    Retorna:
    - Probabilidad de que la URL sea phishing.
    """
    # Convertir las características a un DataFrame de pandas con los nombres de columna
    caracteristicas_df = pd.DataFrame([caracteristicas], columns=nombres_columnas)
    
    # Escalar las características
    caracteristicas_escaladas = scaler.transform(caracteristicas_df)

    # Realizar la predicción
    probabilidad = model.predict(caracteristicas_escaladas)[0][0]  # Probabilidad de clase positiva (phishing)

    return probabilidad

# Nombres de las características seleccionadas
nombres_columnas = [
    'google_index', 'nb_qm', 'nb_www', 'ratio_digits_host', 'nb_hyperlinks',
    'nb_hyphens', 'domain_in_title', 'nb_dots', 'nb_eq', 'length_hostname',
    'domain_age', 'ratio_digits_url', 'domain_with_copyright', 'https_token',
    'whois_registered_domain', 'tld_in_subdomain', 'dns_record', 'prefix_suffix'
]

# Características manualmente ingresadas (valores hipotéticos)
# Características de ejemplo extraídas de la imagen
caracteristicas_ejemplo = [
    1,  # google_index
    0,  # nb_qm
    0,  # nb_www
    0.000000,  # ratio_digits_host
    7,  # nb_hyperlinks
    0,  # nb_hyphens
    0,  # domain_in_title
    2,  # nb_dots
    1,  # nb_eq
    25,  # length_hostname
    4709,  # domain_age
    0.0,  # ratio_digits_url
    0,  # domain_with_copyright
    1,  # https_token
    1,  # whois_registered_domain
    0,  # tld_in_subdomain
    0,  # dns_record
    0   # prefix_suffix
]

# Calcular la probabilidad de phishing
probabilidad_phishing = predecir_probabilidad_phishing(model, scaler, caracteristicas_ejemplo, nombres_columnas)
print(f"Probabilidad de phishing: {probabilidad_phishing:.2%}")

In [ ]:
pd.crosstab(data_mejorada["status"],data_mejorada["statistical_report"],normalize="index").plot.bar()

In [ ]:
# Combinar las columnas seleccionadas
subset = pd.concat([data["url"], X_new, data["status"]], axis=1)

In [ ]:
subset

In [ ]:
subset.drop(columns=["url"]).to_csv("dataset_entrada.csv", index=False)

In [ ]:
subset[["url","nb_www"]].sample(n=6)

In [ ]:
import pandas as pd
import numpy as np

def predecir_dataset(model, scaler, input_csv, output_csv):
    """
    Lee un archivo CSV con las 14 características seleccionadas y la etiqueta `status`,
    realiza las predicciones y genera un archivo de salida con las características originales,
    `status` y una columna adicional para la predicción.

    Parámetros:
    - model: Modelo entrenado.
    - scaler: Escalador utilizado para normalizar los datos.
    - input_csv: Ruta al archivo CSV de entrada.
    - output_csv: Ruta al archivo CSV de salida.
    """
    # Cargar el dataset
    data = pd.read_csv(input_csv)

    # Separar características (asegúrate de que las columnas coincidan con las esperadas)
    X = data.iloc[:, :-1]  # Todas las columnas excepto la última (status)
    y = data.iloc[:, -1]   # La última columna es `status`

    # Escalar las características
    X_scaled = scaler.transform(X)

    # Realizar las predicciones
    y_pred_probs = model.predict(X_scaled)  # Predicciones en probabilidades
    y_pred = (y_pred_probs > 0.5).astype(int)  # Convertir a clases binarias (0 o 1)

    # Agregar la predicción como una nueva columna
    data['Predicción'] = y_pred
    data['Probabilidad'] = y_pred_probs.flatten()

    # Guardar el archivo de salida
    data.to_csv(output_csv, index=False)

    print(f"Archivo de salida guardado en: {output_csv}")

# Ejemplo de uso:
# Define las rutas de entrada y salida
input_csv = 'dataset_entrada.csv'  # Reemplaza con tu archivo CSV de entrada
output_csv = 'dataset_salida.csv'  # Archivo de salida con las predicciones

# Llamar al método para predecir
predecir_dataset(model, scaler, input_csv, output_csv)


In [ ]:
########################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################
########################################################################################################################################################################################################################################################################################################################################################################
####################################################################################################################################################################################################################################################################################################################################
########################################################################################################################################################################################################################################################################################################################################################################

In [ ]:
pd.crosstab(data["status"],data["nb_slash"],normalize="index")

In [ ]:
import pandas as pd

# Obtener las predicciones del modelo
y_pred = (model.predict(X_test) > 0.5).astype(int)  # Convertir probabilidades a clases binarias

# Crear un DataFrame con etiquetas reales y predicciones
resultados = pd.DataFrame({
    'Etiqueta Original': y_test.values,  # Valores originales
    'Predicción': y_pred.flatten()       # Predicciones del modelo
})

# Mostrar las primeras filas del DataFrame
print(resultados.head())

# Opcional: Guardar los resultados en un archivo CSV para análisis posterior
resultados.to_csv('resultados_comparacion.csv', index=False)


In [ ]:
# Asegurarse de usar todo el conjunto de datos original
X_total = scaler.transform(X_new)  # Escalar todo el conjunto de datos original

# Realizar las predicciones en el dataset completo
y_pred_total = (model.predict(X_total) > 0.5).astype(int)  # Convertir probabilidades a clases binarias

# Crear un DataFrame con las características originales del dataset completo
resultados_completos = pd.DataFrame(X, columns=mejores_columnas_XGB)  # Usar las columnas originales seleccionadas

# Agregar las etiquetas originales y las predicciones
resultados_completos['Etiqueta Original'] = y.values  # Agregar las etiquetas originales
resultados_completos['Predicción'] = y_pred_total.flatten()  # Agregar las predicciones del modelo

# Mostrar las primeras filas del DataFrame
print(resultados_completos.head())

# Guardar los resultados en un archivo CSV para análisis posterior
resultados_completos.to_csv('resultados_completos.csv', index=False)


In [ ]:
data_filtrada[mejores_columnas_XGB]

In [ ]:
############################################################################################################################################################################################################################################################################################################################################################
###############################################################################################################################################################################################################################################################################################################################
##################################################################################################################################################################################################################################################################################################
###############################################################################################################################################################################################################################################################################################################################

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Cargar dataset
# Asegúrate de que tu dataset tiene las columnas "url" y "status" (etiqueta)
data2 = pd.read_csv("dataset_phishing.csv")

# Convertir las etiquetas a valores numéricos
label_encoder = LabelEncoder()
data2['status'] = label_encoder.fit_transform(data2['status'])  # 0: legitimate, 1: phishing

# Configurar el Tokenizer para convertir URLs en secuencias de caracteres
tokenizer = Tokenizer(char_level=True)  # Procesa las URLs como secuencias de caracteres
tokenizer.fit_on_texts(data2['url'])

# Convertir URLs en secuencias numéricas
sequences = tokenizer.texts_to_sequences(data2['url'])
max_sequence_length = max(len(seq) for seq in sequences)  # Longitud máxima de secuencia
X2 = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')  # Rellenar secuencias

# Etiquetas
y2 = data2['status']

# Dividir en conjuntos de entrenamiento y prueba
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=42)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Parámetros
vocab_size = len(tokenizer.word_index) + 1  # Número de caracteres únicos en el vocabulario
embedding_dim = 50  # Dimensión de los embeddings

In [ ]:
# Construir modelo RNN
model_rnn = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_sequence_length),
    LSTM(32, return_sequences=False),  # LSTM con 64 unidades
    Dropout(0.5),
    Dense(16, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Salida binaria
])

model_rnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenar modelo RNN
history_rnn = model_rnn.fit(X_train2, y_train2, epochs=10, batch_size=32, validation_split=0.2, verbose=1)


In [ ]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten

# Construir modelo CNN
model_cnn = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_sequence_length),
    Conv1D(filters=128, kernel_size=5, activation='relu'),  # Convolución 1D
    MaxPooling1D(pool_size=2),  # Max pooling
    Flatten(),  # Aplanar
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Salida binaria
])

In [ ]:
model_cnn.summary()

In [ ]:
model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenar modelo CNN
history_cnn = model_cnn.fit(X_train2, y_train2, epochs=10, batch_size=32, validation_split=0.2, verbose=1)


In [ ]:
# Evaluar CNN
loss_cnn, accuracy_cnn = model_cnn.evaluate(X_test2, y_test2, verbose=0)
print(f"Precisión del modelo CNN: {accuracy_cnn:.2%}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Crear el modelo de Random Forest
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)

# Entrenar el modelo con el conjunto de entrenamiento
random_forest.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred2 = random_forest.predict(X_test)

# Evaluar el modelo
accuracy2 = accuracy_score(y_test, y_pred2)
print(f"Exactitud del modelo: {accuracy:.2f}")

# Mostrar el reporte de clasificación
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred2))

# Matriz de confusión
conf_matrix = confusion_matrix(y_test, y_pred2)
print("\nMatriz de confusión:")
print(conf_matrix)

# Importancia de características
importancias = random_forest.feature_importances_
print("\nImportancia de las características:")
for nombre, importancia in zip(X_new.columns, importancias):
    print(f"{nombre}: {importancia:.4f}")


In [ ]:
import csv
# Leer las URLs desde el archivo CSV
archivo_csv = "urls_extraidas.csv"
urls = []

with open(archivo_csv, 'r', encoding='utf-8') as file:
    reader = csv.reader(file)
    for row in reader:
        if row:  # Asegurarse de que no haya filas vacías
            urls.append(row[0])

# Procesar cada URL y calcular su probabilidad de phishing
resultados = []

for url in urls:
    try:
        # Calcular la probabilidad de phishing para la URL
        probabilidad = predecir_probabilidad_phishing_url(model, scaler, url, nombres_columnas)
        resultados.append((url, probabilidad))
    except Exception as e:
        print(f"Error procesando URL: {url} - {e}")
        resultados.append((url, None))  # Si hay error, marcar como None

# Guardar los resultados en un nuevo archivo CSV
archivo_resultados = "resultados_probabilidades.csv"

with open(archivo_resultados, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(["URL", "Probabilidad de Phishing"])  # Escribir encabezado
    for url, probabilidad in resultados:
        writer.writerow([url, probabilidad if probabilidad is not None else "Error"])

print(f"Resultados guardados en '{archivo_resultados}'.")